# Resume Training From Latest Checkpoint

Notebook minimal pour reprendre l'entraînement ST-CDGM depuis le **dernier checkpoint détecté automatiquement**.

Comme `st_cdgm_validation_inference.ipynb` : les poids viennent du `.pt`, mais l’**architecture** (méta-chemins encodeur, UNet, etc.) est dérivée du **YAML** (`config/training_config.yaml` par défaut), fusionné avec `checkpoint["config"]` si présent. Cela évite l’erreur lorsque le checkpoint ne contient pas `encoder.metapaths`.

Ce notebook:
- charge le checkpoint le plus récent (`.pt/.pth/.ckpt`),
- fusionne la config YAML + config embarquée, reconstruit encodeur / RCN / diffusion puis charge les `state_dict`,
- optimiseur **comme `st_cdgm_training_evaluation.ipynb`** : uniquement encoder + RCN + diffusion (pas de `SpatialConditioningProjector` dans l’optim ni dans `train_epoch`),
- restaure l’optimizer si disponible et compatible (sinon message + état neuf),
- reprend l’entraînement,
- sauvegarde un checkpoint à chaque epoch + un final.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional

# --- Racine du dépôt (aligné sur st_cdgm_training_evaluation / validation_inference) ---
project_root = Path.cwd()
if not (project_root / "src" / "st_cdgm").is_dir():
    for p in [project_root, *project_root.parents]:
        if (p / "src" / "st_cdgm").is_dir():
            project_root = p.resolve()
            break
try:
    os.chdir(project_root)
except OSError:
    pass

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))
# scripts/ et ops/ sont à la racine du dépôt
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from omegaconf import OmegaConf

from st_cdgm import (
    NetCDFDataPipeline,
    HeteroGraphBuilder,
    RCNCell,
    RCNSequenceRunner,
    CausalDiffusionDecoder,
    train_epoch,
)
from ops.train_st_cdgm import build_encoder_for_graph

print(f"cwd: {os.getcwd()}")
print(f"PYTHONPATH src: {str(project_root / 'src') in sys.path}")
print(f"project_root: {project_root}")
print("Imports OK")

In [ ]:
# ---- User parameters ----
ROOT = project_root
CHECKPOINT_SEARCH_DIRS = [
    ROOT / 'models',
    ROOT / 'checkpoints',
    ROOT,
]
CHECKPOINT_EXTS = ('.pt', '.pth', '.ckpt')

# Nombre d'epochs SUPPLEMENTAIRES à entraîner après reprise
RESUME_EPOCHS = 5

# Sauvegarde
SAVE_EVERY_EPOCH = 1
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = ROOT / 'models' / 'resume_runs' / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config YAML = squelette d'architecture (comme st_cdgm_validation_inference).
# Changer en training_config_vice.yaml si votre entraînement utilise ce fichier.
CONFIG_YAML = ROOT / 'config' / 'training_config.yaml'
CONFIG = OmegaConf.load(CONFIG_YAML)

_df = str(CONFIG.data.get('dataset_format', 'netcdf')).lower()
print(f'dataset_format (CONFIG): {_df}')
if _df != 'netcdf':
    print(
        '[WARN] Ce notebook lit les donnees via NetCDFDataPipeline (.nc). '
        'Utilisez data.dataset_format: netcdf dans le YAML si vous n\'utilisez pas zarr/shards (train_ddp).'
    )

# Device (comme validation_inference : training.device + dispo CUDA)
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() and CONFIG.training.device == 'cuda' else 'cpu'
)
print(f'Device: {DEVICE}')
print(f'Output dir: {OUTPUT_DIR}')


def find_latest_checkpoint(
    search_dirs: List[Path],
    exts: tuple[str, ...] = CHECKPOINT_EXTS,
) -> Path:
    candidates: List[Path] = []
    for base in search_dirs:
        if not base.exists():
            continue
        for ext in exts:
            candidates.extend(base.rglob(f'*{ext}'))

    # Exclure checkpoints metadata éventuels ou fichiers temporaires
    candidates = [p for p in candidates if p.is_file() and not p.name.endswith('.tmp')]

    if not candidates:
        searched = '\n'.join(str(p) for p in search_dirs)
        raise FileNotFoundError(
            'Aucun checkpoint trouvé. Dossiers scannés:\n' + searched
        )

    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    return latest


LATEST_CHECKPOINT = find_latest_checkpoint(CHECKPOINT_SEARCH_DIRS)
print(f'Latest checkpoint selected: {LATEST_CHECKPOINT}')

In [ ]:
# Chargement brut + fusion config (même logique que st_cdgm_validation_inference :
# le .pt peut contenir un `config` partiel / plat sans encoder.metapaths ; le YAML complète.)
try:
    raw_ckpt = torch.load(Path(LATEST_CHECKPOINT), map_location=DEVICE, weights_only=False)
except TypeError:
    raw_ckpt = torch.load(Path(LATEST_CHECKPOINT), map_location=DEVICE)

_required = {'encoder_state_dict', 'rcn_cell_state_dict', 'diffusion_state_dict'}
_missing = _required - raw_ckpt.keys()
if _missing:
    raise KeyError(f'Checkpoint invalide, clés manquantes: {_missing}')

import copy

ckpt_cfg_dict = raw_ckpt.get('config')
if ckpt_cfg_dict is None:
    ckpt_cfg_dict = {}
else:
    # Ne jamais laisser le .pt ecraser les listes de variables (souvent [] ou absentes) : cause LR [..., 0, H, W] -> 598x0
    ckpt_cfg_dict = copy.deepcopy(
        OmegaConf.to_container(OmegaConf.create(ckpt_cfg_dict), resolve=True)
    )
    _data = ckpt_cfg_dict.get('data')
    if isinstance(_data, dict):
        for _k in ('lr_variables', 'hr_variables', 'static_variables'):
            _data.pop(_k, None)

cfg = OmegaConf.merge(CONFIG, OmegaConf.create(ckpt_cfg_dict))

# Si le checkpoint n'embarque pas la liste des méta-chemins (souvent le cas), garder encoder du YAML
_mpaths = cfg.get('encoder', {}).get('metapaths')
if not _mpaths:
    cfg.encoder = CONFIG.encoder
    print('[resume] encoder.metapaths absent ou vide dans le checkpoint — utilisation du YAML (CONFIG).')

start_epoch = int(raw_ckpt.get('epoch', 0))
optimizer_state_dict = raw_ckpt.get('optimizer_state_dict')

print(f'Start epoch in checkpoint: {start_epoch}')
print(f'Config: OmegaConf.merge(CONFIG, checkpoint["config"]) — YAML: {CONFIG_YAML}')

In [ ]:
# Rebuild data pipeline : listes LR/HR/static **uniquement depuis CONFIG** (YAML charge en cellule 2).
# Les chemins data.* viennent de cfg (checkpoint + YAML) ; les noms de variables ne doivent pas venir du .pt.
_lr_vars = list(CONFIG.data.lr_variables)
_hr_vars = list(CONFIG.data.hr_variables)
_st_vars = list(CONFIG.data.static_variables)
print(f'[resume] Pipeline variables (CONFIG): lr={len(_lr_vars)} hr={len(_hr_vars)} static={len(_st_vars)}')

_mean = getattr(cfg.data, 'means_path', None) or getattr(cfg.data, 'mean_path', None)
_std = getattr(cfg.data, 'stds_path', None) or getattr(cfg.data, 'std_path', None)
if not _mean:
    _mean = str(ROOT / 'data' / 'raw' / 'normalization_coefs' / 'mean_1974_2011.nc')
if not _std:
    _std = str(ROOT / 'data' / 'raw' / 'normalization_coefs' / 'std_1974_2011.nc')
_mean = _mean if Path(_mean).exists() else None
_std = _std if Path(_std).exists() else None

pipeline = NetCDFDataPipeline(
    lr_path=cfg.data.lr_path,
    hr_path=cfg.data.hr_path,
    static_path=cfg.data.static_path,
    seq_len=cfg.data.seq_len,
    baseline_strategy=cfg.data.baseline_strategy,
    baseline_factor=cfg.data.baseline_factor,
    target_transform=cfg.data.get('target_transform'),
    normalize=cfg.data.normalize,
    nan_fill_strategy=getattr(cfg.data, 'nan_fill_strategy', 'zero'),
    precipitation_delta=getattr(cfg.data, 'precipitation_delta', 0.01),
    lr_variables=_lr_vars or None,
    hr_variables=_hr_vars or None,
    static_variables=_st_vars or None,
    means_path=_mean,
    stds_path=_std,
)

sample_ds = pipeline.build_sequence_dataset(
    seq_len=cfg.data.seq_len,
    stride=cfg.data.stride,
    as_torch=True,
)
sample_for_shapes = next(iter(sample_ds))
assert sample_for_shapes['lr'].shape[1] > 0, (
    'LR a 0 canaux — verifiez cfg/CONFIG.data.lr_variables et le fichier lr_path.'
)
rcn_driver_dim = sample_for_shapes['lr'].shape[1]
hr_channels = sample_for_shapes['residual'].shape[1]

train_dataset = pipeline.build_sequence_dataset(
    seq_len=cfg.data.seq_len,
    stride=cfg.data.stride,
    as_torch=True,
)

# DataLoader batch-aware (comme st_cdgm_training_evaluation.ipynb)
BATCH_SIZE = int(cfg.training.get('batch_size', 1))
NUM_WORKERS = int(cfg.training.get('num_workers', 0))
if NUM_WORKERS < 0:
    NUM_WORKERS = 0
PIN_MEMORY = bool(DEVICE.type == 'cuda')
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
PERSISTENT_WORKERS = NUM_WORKERS > 0

_loader_kwargs = dict(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    collate_fn=lambda x: x,
)
if PREFETCH_FACTOR is not None:
    _loader_kwargs['prefetch_factor'] = PREFETCH_FACTOR
train_dataloader = DataLoader(**_loader_kwargs)
print(f"[resume] DataLoader batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")


def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]

    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)

    dynamic_features = {node_type: lr_nodes_steps[0] for node_type in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)

    out = {
        'lr': lr_tensor,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero': hetero,
    }
    if 'valid_mask' in sample:
        out['valid_mask'] = sample['valid_mask']
    return out


def iterate_batches(dataloader, builder, device):
    """Compatible train_epoch: yield dict ou liste de dicts (micro-batches)."""
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        converted = [convert_sample_to_batch(sample, builder, device) for sample in batch_list]
        yield converted if len(converted) > 1 else converted[0]


builder = HeteroGraphBuilder(
    lr_shape=tuple(cfg.graph.lr_shape),
    hr_shape=tuple(cfg.graph.hr_shape),
    static_dataset=pipeline.get_static_dataset(),
    # IMPORTANT: utiliser les variables statiques du dataset (CONFIG.data.static_variables).
    # cfg.graph.static_variables est vide dans le YAML ([]) et donnerait SP_HR avec 0 canaux -> matmul 598x0.
    static_variables=_st_vars or None,
    include_mid_layer=cfg.graph.include_mid_layer,
)

# Reconstruction des modules puis chargement des poids (aligné validation_inference + ops/train_st_cdgm)
encoder = build_encoder_for_graph(cfg.encoder, builder).to(DEVICE).train()

try:
    from st_cdgm.utils.checkpoint import strip_torch_compile_prefix
except ModuleNotFoundError:
    def strip_torch_compile_prefix(sd):
        if not sd:
            return sd
        if not any(str(k).startswith('_orig_mod.') for k in sd.keys()):
            return sd
        return {
            (k[len('_orig_mod.') :] if str(k).startswith('_orig_mod.') else k): v
            for k, v in sd.items()
        }

rcn_cell = RCNCell(
    num_vars=len(encoder.configs),
    hidden_dim=cfg.rcn.hidden_dim,
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=cfg.rcn.dropout,
).to(DEVICE).train()

rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=cfg.rcn.get('detach_interval'))

# Dimension d'embedding classe UNet : doit coller au checkpoint (ex. 768 vs 640 si 6 vs 5 méta-chemins)
_diff_sd = strip_torch_compile_prefix(raw_ckpt['diffusion_state_dict'])
_proj_in_ckpt = None
for _k, _v in _diff_sd.items():
    if str(_k).endswith('class_embedding.linear_1.weight'):
        _proj_in_ckpt = int(_v.shape[1])
        break

unet_kwargs = dict(cfg.diffusion.unet_kwargs) if cfg.diffusion.get('unet_kwargs') else dict(
    layers_per_block=1,
    block_out_channels=(32, 64),
    down_block_types=('DownBlock2D', 'CrossAttnDownBlock2D'),
    up_block_types=('CrossAttnUpBlock2D', 'UpBlock2D'),
    mid_block_type='UNetMidBlock2D',
    norm_num_groups=8,
    class_embed_type='projection',
    projection_class_embeddings_input_dim=len(encoder.configs) * cfg.diffusion.conditioning_dim,
    resnet_time_scale_shift='scale_shift',
    attention_head_dim=32,
    only_cross_attention=[False, True],
)
if _proj_in_ckpt is not None:
    unet_kwargs = dict(unet_kwargs)
    _proj_yaml = unet_kwargs.get('projection_class_embeddings_input_dim')
    unet_kwargs['projection_class_embeddings_input_dim'] = _proj_in_ckpt
    if _proj_yaml is not None and _proj_yaml != _proj_in_ckpt:
        print(
            f'[resume] projection_class_embeddings_input_dim: YAML {_proj_yaml} -> checkpoint {_proj_in_ckpt}'
        )
    _expected = len(encoder.configs) * int(cfg.diffusion.conditioning_dim)
    if _proj_in_ckpt != _expected:
        print(
            f'[resume] Attention: dim class embedding UNet ({_proj_in_ckpt}) != '
            f'len(encoder.configs)*conditioning_dim ({_expected}). Verifiez graphe / metapaths.'
        )

diffusion = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=cfg.diffusion.conditioning_dim,
    height=cfg.diffusion.height,
    width=cfg.diffusion.width,
    num_diffusion_steps=cfg.diffusion.steps,
    unet_kwargs=unet_kwargs,
    scheduler_type=cfg.diffusion.get('scheduler_type', 'ddpm'),
    use_gradient_checkpointing=cfg.diffusion.get('use_gradient_checkpointing', False),
    conv_padding_mode=cfg.diffusion.get('conv_padding_mode', 'zeros'),
    anti_checkerboard=cfg.diffusion.get('anti_checkerboard', False),
).to(DEVICE).train()

encoder.load_state_dict(strip_torch_compile_prefix(raw_ckpt['encoder_state_dict']))
rcn_cell.load_state_dict(strip_torch_compile_prefix(raw_ckpt['rcn_cell_state_dict']))
diffusion.load_state_dict(_diff_sd)

# Optimizer : meme composition que st_cdgm_training_evaluation.ipynb (encoder + RCN + diffusion uniquement)
params = (
    list(encoder.parameters())
    + list(rcn_cell.parameters())
    + list(diffusion.parameters())
)
optimizer = optim.Adam(params, lr=cfg.training.lr)
if optimizer_state_dict is not None:
    try:
        optimizer.load_state_dict(optimizer_state_dict)
        print('Optimizer state restored from checkpoint')
    except ValueError as e:
        print(
            '[resume] Impossible de charger optimizer_state_dict (groupes/parametres differents). '
            f'Optimiseur reinitialise. Detail: {e}'
        )
else:
    print('No optimizer_state_dict in checkpoint; using fresh optimizer')

print(f'Inferred dims: driver_dim={rcn_driver_dim}, hr_channels={hr_channels}')

In [ ]:
def save_resume_checkpoint(
    out_dir: Path,
    epoch: int,
    metrics: Dict[str, float],
    *,
    encoder,
    rcn_cell,
    diffusion,
    optimizer,
    cfg,
) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    ckpt_path = out_dir / f'resume_epoch_{epoch:04d}_{ts}.pt'

    payload = {
        'epoch': int(epoch),
        'encoder_state_dict': encoder.state_dict(),
        'rcn_cell_state_dict': rcn_cell.state_dict(),
        'diffusion_state_dict': diffusion.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': {k: float(v) for k, v in metrics.items()},
        'config': OmegaConf.to_container(cfg, resolve=True),
        'source_checkpoint': str(LATEST_CHECKPOINT),
        'saved_at': ts,
    }
    torch.save(payload, ckpt_path)
    return ckpt_path

In [ ]:
history: List[Dict[str, float]] = []
last_ckpt_path: Optional[Path] = None

end_epoch = start_epoch + RESUME_EPOCHS
print(f'Resuming training from epoch {start_epoch + 1} to {end_epoch}')

for epoch in range(start_epoch, end_epoch):
    print('\n' + '=' * 80)
    print(f'Epoch {epoch + 1}/{end_epoch}')
    print('=' * 80)

    batch_iter = iterate_batches(train_dataloader, builder, DEVICE)

    # Memes arguments que st_cdgm_training_evaluation.ipynb (boucle train_epoch)
    metrics = train_epoch(
        encoder=encoder,
        rcn_runner=rcn_runner,
        diffusion_decoder=diffusion,
        optimizer=optimizer,
        data_loader=batch_iter,
        lambda_gen=cfg.loss.lambda_gen,
        beta_rec=cfg.loss.beta_rec,
        gamma_dag=cfg.loss.gamma_dag,
        conditioning_fn=None,
        device=DEVICE,
        use_amp=cfg.training.get('use_amp', True),
        gradient_clipping=cfg.training.get('gradient_clipping', None),
        log_interval=cfg.training.get('log_every', 10),
        dag_method=cfg.loss.get('dag_method', 'dagma'),
        dagma_s=cfg.loss.get('dagma_s', 1.0),
        reconstruction_loss_type=cfg.loss.get('reconstruction_loss_type', 'mse'),
    )

    history.append({'epoch': epoch + 1, **{k: float(v) for k, v in metrics.items()}})

    if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
        last_ckpt_path = save_resume_checkpoint(
            OUTPUT_DIR,
            epoch=epoch + 1,
            metrics=metrics,
            encoder=encoder,
            rcn_cell=rcn_cell,
            diffusion=diffusion,
            optimizer=optimizer,
            cfg=cfg,
        )
        print(f'Checkpoint saved: {last_ckpt_path}')

print('\nTraining resume finished.')
print(f'Last checkpoint: {last_ckpt_path}')

In [ ]:
# Smoke validation: check checkpoint created
created = sorted(OUTPUT_DIR.glob('resume_epoch_*.pt'))
print(f'Created checkpoint files: {len(created)}')
if created:
    print('Latest:', created[-1])
else:
    raise RuntimeError(f'No checkpoint created in {OUTPUT_DIR}')

# Optional: save one explicit final checkpoint
final_metrics = history[-1] if history else {'loss': float('nan')}
final_ckpt = save_resume_checkpoint(
    OUTPUT_DIR,
    epoch=(history[-1]['epoch'] if history else start_epoch),
    metrics=final_metrics,
    encoder=encoder,
    rcn_cell=rcn_cell,
    diffusion=diffusion,
    optimizer=optimizer,
    cfg=cfg,
)
print('Final checkpoint:', final_ckpt)